In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import  mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/raw/india_agriculture_seed_sales_data.csv')

 Model 1: Sales Forecasting

In [2]:
le = LabelEncoder()
df_ml = df.copy()
for col in ['Region','State','Vegetable_Type']:
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col])

features = ['Year','Month','Region_enc','State_enc','Vegetable_Type_enc',
            'Warehouse_Quantity','Competitor_Market_Share_%',
            'Farmer_Sentiment_Score','Seed_Quality_Index_%',
            'Rainfall_mm','Temperature_C','Irrigation_Access_%',
            'Pest_Infestation_Index','Seed_Price_Per_Unit',
            'Sales_Target','Soil_pH','Soil_Nitrogen_ppm']

# Split the data in train and test
X = df_ml[features]
y = df_ml['Actual_Units_Sold']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


In [3]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoosting' : XGBRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = {}
predictions_dict = {}

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Calculate metrics
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

 # Cross-validation score
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    results[name] = {
        'Train R2': train_r2,
        'Test R2': test_r2,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'CV Mean R2': cv_scores.mean(),
        'CV Std R2': cv_scores.std()
    }
    predictions_dict[name] = y_pred_test

    print(f"\n{'='*40}")
    print(f"{name} Results:")
    print(f"{'='*40}")
    print(f"Train R² Score: {train_r2:.4f}")
    print(f"Test R² Score: {test_r2:.4f}")
    print(f"Train MAE: {train_mae:,.0f}")
    print(f"Test MAE: {test_mae:,.0f}")
    print(f"Train RMSE: {train_rmse:,.0f}")
    print(f"Test RMSE: {test_rmse:,.0f}")
    print(f"5-Fold CV R² Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")


Linear Regression Results:
Train R² Score: 0.9801
Test R² Score: 0.9802
Train MAE: 58
Test MAE: 59
Train RMSE: 77
Test RMSE: 78
5-Fold CV R² Mean: 0.9800 (+/- 0.0018)

Ridge Regression Results:
Train R² Score: 0.9801
Test R² Score: 0.9802
Train MAE: 58
Test MAE: 59
Train RMSE: 77
Test RMSE: 78
5-Fold CV R² Mean: 0.9800 (+/- 0.0018)

Lasso Regression Results:
Train R² Score: 0.9801
Test R² Score: 0.9802
Train MAE: 58
Test MAE: 59
Train RMSE: 77
Test RMSE: 78
5-Fold CV R² Mean: 0.9800 (+/- 0.0018)

Random Forest Results:
Train R² Score: 0.9970
Test R² Score: 0.9784
Train MAE: 22
Test MAE: 59
Train RMSE: 30
Test RMSE: 81
5-Fold CV R² Mean: 0.9786 (+/- 0.0018)

XGBoosting Results:
Train R² Score: 0.9938
Test R² Score: 0.9751
Train MAE: 33
Test MAE: 59
Train RMSE: 43
Test RMSE: 87
5-Fold CV R² Mean: 0.9775 (+/- 0.0024)

Gradient Boosting Results:
Train R² Score: 0.9825
Test R² Score: 0.9803
Train MAE: 55
Test MAE: 58
Train RMSE: 72
Test RMSE: 78
5-Fold CV R² Mean: 0.9800 (+/- 0.0020)


Here Linear Regression is the best model because Test R² (0.9802) is actually higher than Train R² (0.9801), This indicates no overfitting whatsoever. The model generalizes perfectly to new data

Feature Importance

In [19]:
# Access the trained Linear Regression model
lr_model = models['Linear Regression']

# Get coefficients (these are like feature importances for linear models)
feature_importance = pd.Series(
    lr_model.coef_,  # ✅ Use .coef_ instead of .feature_importances_
    index=X2.columns  # Use your feature names
).sort_values(key=abs, ascending=False)  # Sort by absolute value

# Display top 10 features
print("\n" + "="*50)
print("TOP 10 MOST INFLUENTIAL FEATURES (Linear Regression)")
print("="*50)
for i, (feature, coef) in enumerate(feature_importance.head(10).items(), 1):
    print(f"{i:2d}. {feature:30s} {coef:+.4f}")  # + shows positive/negative impact

# Also display with interpretation
print("\n" + "="*60)
print("FEATURE INTERPRETATION (Linear Regression)")
print("="*60)
print("Positive coefficient: Increases Profit Margin")
print("Negative coefficient: Decreases Profit Margin")
print("Larger absolute value = Stronger impact")
print("-"*60)


TOP 10 MOST INFLUENTIAL FEATURES (Linear Regression)
 1. Pest_Infestation_Index         -4.6231
 2. Days_to_Sell_Inventory         -3.7898
 3. Farmer_Sentiment_Score         +1.2130
 4. Region_enc                     -0.5919
 5. Year                           -0.1620
 6. Temperature_C                  +0.1501
 7. Seed_Price_Per_Unit            +0.1021
 8. Vegetable_Type_enc             +0.0685
 9. Seed_Quality_Index_%           +0.0660
10. Month                          +0.0360

FEATURE INTERPRETATION (Linear Regression)
Positive coefficient: Increases Profit Margin
Negative coefficient: Decreases Profit Margin
Larger absolute value = Stronger impact
------------------------------------------------------------


Save model 

In [20]:
import joblib
joblib.dump(rf_model, '../outputs/models/sales_forecast_model.pkl')

['../outputs/models/sales_forecast_model.pkl']

 Model 2: Profit Margin Prediction

In [13]:
features_profit = ['Year','Month','Region_enc','Vegetable_Type_enc',
                   'Competitor_Market_Share_%','Farmer_Sentiment_Score',
                   'Seed_Quality_Index_%','Rainfall_mm','Temperature_C',
                   'Irrigation_Access_%','Pest_Infestation_Index',
                   'Seed_Price_Per_Unit','Wholesale_Price','Days_to_Sell_Inventory']

X2 = df_ml[features_profit]
y2 = df_ml['Profit_Margin_%']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42)


In [16]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoosting' : XGBRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = {}
predictions_dict = {}

for name, model2 in models.items():
    # Train
    model2.fit(X2_train, y2_train)

    # Predict
    y2_pred_train = model2.predict(X2_train)
    y2_pred_test = model2.predict(X2_test)

    # Calculate metrics
    train_r2 = r2_score(y2_train, y2_pred_train)
    test_r2 = r2_score(y2_test, y2_pred_test)
    train_mae = mean_absolute_error(y2_train, y2_pred_train)
    test_mae = mean_absolute_error(y2_test, y2_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y2_train, y2_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_test))

 # Cross-validation score
    cv_scores = cross_val_score(model2, X2_train, y2_train, cv=5, scoring='r2')

    results[name] = {
        'Train R2': train_r2,
        'Test R2': test_r2,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'CV Mean R2': cv_scores.mean(),
        'CV Std R2': cv_scores.std()
    }
    predictions_dict[name] = y2_pred_test

    print(f"\n{'='*40}")
    print(f"{name} Results:")
    print(f"{'='*40}")
    print(f"Train R² Score: {train_r2:.4f}")
    print(f"Test R² Score: {test_r2:.4f}")
    print(f"Train MAE: {train_mae:,.0f}")
    print(f"Test MAE: {test_mae:,.0f}")
    print(f"Train RMSE: {train_rmse:,.0f}")
    print(f"Test RMSE: {test_rmse:,.0f}")
    print(f"5-Fold CV R² Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")


Linear Regression Results:
Train R² Score: 0.5211
Test R² Score: 0.5118
Train MAE: 12
Test MAE: 12
Train RMSE: 16
Test RMSE: 16
5-Fold CV R² Mean: 0.5198 (+/- 0.0163)

Ridge Regression Results:
Train R² Score: 0.5211
Test R² Score: 0.5118
Train MAE: 12
Test MAE: 12
Train RMSE: 16
Test RMSE: 16
5-Fold CV R² Mean: 0.5198 (+/- 0.0163)

Lasso Regression Results:
Train R² Score: 0.5186
Test R² Score: 0.5075
Train MAE: 12
Test MAE: 12
Train RMSE: 16
Test RMSE: 16
5-Fold CV R² Mean: 0.5176 (+/- 0.0163)

Random Forest Results:
Train R² Score: 0.9519
Test R² Score: 0.6454
Train MAE: 4
Test MAE: 10
Train RMSE: 5
Test RMSE: 14
5-Fold CV R² Mean: 0.6538 (+/- 0.0234)

XGBoosting Results:
Train R² Score: 0.8496
Test R² Score: 0.6287
Train MAE: 7
Test MAE: 10
Train RMSE: 9
Test RMSE: 14
5-Fold CV R² Mean: 0.6372 (+/- 0.0155)

Gradient Boosting Results:
Train R² Score: 0.6882
Test R² Score: 0.6601
Train MAE: 10
Test MAE: 10
Train RMSE: 13
Test RMSE: 13
5-Fold CV R² Mean: 0.6678 (+/- 0.0234)


Best model is Gradient Boosting, because
Highest Test R² (0.6601) - Explains 66% of variance in unseen data.
Smallest Train-Test Gap (0.0281) - Very little overfitting.
Highest CV Mean (0.6678) - Consistent performance across folds

Classify Profitable vs Loss-Making

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [23]:
df_ml['Profit_Class'] = (df_ml['Profit_Margin_%'] > 0).astype(int)
y3 = df_ml['Profit_Class']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X2, y3, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X3_train, y3_train)
print("\nProfit Classification Report:")
print(classification_report(y3_test, clf.predict(X3_test)))


Profit Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.74      0.76      1718
           1       0.83      0.86      0.85      2602

    accuracy                           0.81      4320
   macro avg       0.81      0.80      0.80      4320
weighted avg       0.81      0.81      0.81      4320



Model 3: Farmer Churn Risk Score

In [29]:
from sklearn.tree import DecisionTreeClassifier   # Import Decision Tree Classifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import  XGBClassifier
from sklearn import metrics
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [25]:
df_ml['Churn_Risk'] = (
    (df_ml['Farmer_Sentiment_Score'] < df_ml['Farmer_Sentiment_Score'].quantile(0.3)) &
    (df_ml['Competitor_Market_Share_%'] > df_ml['Competitor_Market_Share_%'].quantile(0.7))
).astype(int)

print(f"Churn Risk Rate: {df_ml['Churn_Risk'].mean():.1%}")

Churn Risk Rate: 8.9%


In [31]:
churn_features = ['Farmer_Sentiment_Score','Seed_Quality_Index_%',
                  'Competitor_Market_Share_%','Pest_Infestation_Index',
                  'Rainfall_mm','Irrigation_Access_%','Farmer_Share_%',
                  'Days_to_Sell_Inventory','Region_enc','Season_enc' if 'Season_enc' in df_ml.columns else 'Month']

churn_features = [f for f in churn_features if f in df_ml.columns]
X4 = df_ml[churn_features]
y4 = df_ml['Churn_Risk']

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y4, test_size=0.2, random_state=42)

from sklearn.ensemble import GradientBoostingClassifier
churn_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
churn_model.fit(X4_train, y4_train)

df_ml['Churn_Risk_Score'] = churn_model.predict_proba(X4)[:,1]

# High-risk segments
high_risk = df_ml[df_ml['Churn_Risk_Score'] > 0.7].groupby(['Region','State']).size()
print("\nHighest Churn Risk Areas:")
print(high_risk.sort_values(ascending=False).head(8))



Highest Churn Risk Areas:
Region   State         
Central  Madhya Pradesh    131
East     Bihar             130
Central  Uttarakhand       125
         Chhattisgarh      116
East     Odisha            116
South    Tamil Nadu        114
East     Jharkhand         113
South    Andhra Pradesh    108
dtype: int64


In [34]:
import joblib
joblib.dump(churn_model, '../outputs/models/churn_risk_model.pkl')

['../outputs/models/churn_risk_model.pkl']